# 第 6 章习题与解答

> 本章习题围绕「完整 forward pass 的 tensor shape」和「权重绑定」展开。

## Exercise 6.1（易）

**题目**:构造一个 `(1, 4)` 的假 `input_ids`,跑一次 `model(input_ids)`,打印 logits 的 shape。解释每个维度的含义。

<details><summary><b>参考答案</b></summary>

In [ ]:
import sys, torch
sys.path.insert(0, '/home/minimind')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

model = MiniMindForCausalLM(MiniMindConfig()).eval()

# 构造假 input_ids:batch=1, seq_len=4
input_ids = torch.tensor([[1, 100, 200, 2]])  # (1, 4)
print(f"input_ids shape: {input_ids.shape}")  # (1, 4)

with torch.no_grad():
    output = model(input_ids)

print(f"logits shape: {output.logits.shape}")  # (1, 4, 6400)
# 第 1 维 (1)   = batch_size
# 第 2 维 (4)   = seq_len（序列长度）
# 第 3 维 (6400) = vocab_size（词表大小，每个位置对每个 token 的打分）

**解释**:
- `(1, 4, 6400)` = `(batch_size, seq_len, vocab_size)`
- `logits[t][v]` 表示位置 t 预测下一个 token 是词表中第 v 个 token 的原始分数
- 取 `logits[0, -1, :]` 就能得到最后一个位置对下一个 token 的预测分布（推理时用的就是它）

</details>

## Exercise 6.2（中）

**题目**:解释 `tie_word_embeddings=True` 和 `=False` 的参数量差异。在 64M 模型上,差异占比是多少?为什么 minimind 选择绑定?

<details><summary><b>参考答案</b></summary>

In [ ]:
import sys, torch
sys.path.insert(0, '/home/minimind')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

config_tied = MiniMindConfig(tie_word_embeddings=True)
config_untied = MiniMindConfig(tie_word_embeddings=False)

n_tied = sum(p.numel() for p in MiniMindForCausalLM(config_tied).parameters())
n_untied = sum(p.numel() for p in MiniMindForCausalLM(config_untied).parameters())

diff = n_untied - n_tied
print(f"tied   参数量: {n_tied:>12,}  ({n_tied/1e6:.2f}M)")
print(f"untied 参数量: {n_untied:>12,}  ({n_untied/1e6:.2f}M)")
print(f"差异         : {diff:>12,}  ({diff/1e6:.2f}M)")
print(f"差异占 tied  : {diff/n_tied*100:.1f}%")
# 差异 = 4,915,200 = 6400 × 768
# 占比 = 7.7%

**分析**:

差异来自 `lm_head`:
- **tied**: `embed_tokens` 和 `lm_head` 共享同一个 `(6400, 768)` 权重矩阵 → `model.parameters()` 只计数一次
- **untied**: `lm_head` 有自己独立的 `(6400, 768)` 权重 → 多出 4,915,200 个参数

差异恰好 = `vocab_size × hidden_size` = `6400 × 768` = 4,915,200。

**为什么 minimind 选择绑定**:
1. **省 7.7% 参数** —— 在 64M 的小模型上,7.7% 是不可忽略的预算
2. **正则化效果** —— 编码(查表)和解码(lm_head)在同一个语义空间里,绑定强制它们一致
3. **与 Qwen3 架构对齐** —— Qwen 系列在小模型上也用 tied embeddings

> 大模型(如 70B+）通常**不绑定**,因为参数预算充足,独立的 lm_head 能提供更好的表达能力。这是规模相关的工程取舍。

</details>

## Exercise 6.3（难）

**题目**:`logits_to_keep` 参数在 RL 训练中如何节省计算?为什么 SFT 不需要它?

<details><summary><b>参考答案</b></summary>

In [ ]:
import sys, torch
sys.path.insert(0, '/home/minimind')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

model = MiniMindForCausalLM(MiniMindConfig()).eval()
input_ids = torch.randint(0, 6400, (1, 100))  # 长序列 100 个 token

# 默认:logits_to_keep=0,计算所有 100 个位置的 logits
with torch.no_grad():
    out_full = model(input_ids, logits_to_keep=0)
print(f"logits_to_keep=0:  logits shape = {out_full.logits.shape}")
# (1, 100, 6400) — 所有位置都算了 lm_head

# logits_to_keep=1:只算最后 1 个位置的 logits
with torch.no_grad():
    out_last = model(input_ids, logits_to_keep=1)
print(f"logits_to_keep=1:  logits shape = {out_last.logits.shape}")
# (1, 1, 6400) — 只算了最后 1 个位置

**分析**:

### RL 训练中为什么有用

在 PPO/GRPO 等 RL 算法中（第 13 章），生成阶段是**自回归的** —— 每一步只需要**最后一个位置**的 logits 来采样下一个 token:

```python
# generate 循环中的每一步
logits = model(input_ids[:, past_len:], past_key_values=kvs, logits_to_keep=1).logits
next_token = sample(logits[:, -1, :])  # 只用最后一个位置
```

`logits_to_keep=1` 让 lm_head 只对最后 1 个位置做 `768→6400` 的投影,跳过其余 `T-1` 个位置的计算。

**节省多少?** lm_head 的 FLOPS ∝ `seq_len × hidden_size × vocab_size`。`logits_to_keep=1` 把 seq_len 从 T 降到 1,**省了 (T-1)/T 的 lm_head 计算**。序列越长,省得越多。

### SFT 为什么不需要

SFT 的 loss 是:
```python
loss = F.cross_entropy(logits[:, :-1].reshape(-1, V), labels[:, 1:].reshape(-1))
```

CE loss 需要**所有位置**的 logits(每个位置都要预测下一个 token)。如果只保留最后 N 个位置,就会丢失大部分训练信号。

所以 SFT 中 `logits_to_keep=0`(默认）,即**保留所有位置**。

> 这也解释了为什么 forward 签名里 `logits_to_keep` 的默认值是 `0` 而非 `None` —— `slice(-0, None)` 等价于 `slice(0, None)`(保留全部),这是一个巧妙的默认值设计。

</details>